In [1]:
# %% [markdown]
# # 07_multi_model_experiment.ipynb
# Runs the full v2 experiment (60 questions x 5 iterations) against
# GPT-4o instead of GPT-4o-mini, using the EXACT SAME pipeline
# (build_v2_prompt, semantic_compiler, evaluate) as
# 04_full_experiment.py -- only the model string changes.
#
# Directly addresses Reviewer 2 Concern #2: "The claim that SL
# maintains accuracy while reducing exposure is model-dependent and
# should be validated on at least one additional model." GPT-4o is
# chosen as the same-family, larger sibling of GPT-4o-mini (the model
# used throughout the rest of this paper), isolating the effect of
# model capability from the effect of model family/vendor.
#
# Saves to a SEPARATE results file (results_phase3_v2_gpt4o.csv) so
# the GPT-4o-mini results (results_phase3_v2_arch.csv) are never
# overwritten or mixed with this run.

# %%
print("SCRIPT VERSION: 2026-07-15-v1")

import json
import pandas as pd
import time
from utils import query_semantic_layer_v2, evaluate, DB_PATH

print(f"DB_PATH in use: {DB_PATH}")

MODEL_UNDER_TEST = "gpt-4o"

with open('credit_rating_questions_all.json', 'r', encoding='utf-8') as f:
    questions = json.load(f)

df_q = pd.DataFrame(questions)
total = len(df_q)
print(f"Total questions: {total}")
if total != 60:
    print(f"!! WARNING: expected 60 questions, found {total}. Check the question file.")

N_ITER = 5
RESULTS_PATH = f'./results_phase3_v2_{MODEL_UNDER_TEST.replace("-", "").replace(".", "")}.csv'
# resolves to results_phase3_v2_gpt4o.csv for MODEL_UNDER_TEST = "gpt-4o"

print(f"Model under test: {MODEL_UNDER_TEST}")
print(f"Iterations per question: {N_ITER}")
print(f"Results will be saved to: {RESULTS_PATH}")
print(f"Total API calls if run from scratch: {total * N_ITER}")

# %% [markdown]
# ## 1. Resume support (identical pattern to 04_full_experiment.py)

# %%
try:
    df_saved = pd.read_csv(RESULTS_PATH)
    results = df_saved.to_dict('records')
    done_ids = set(
        q_id for q_id, grp in df_saved.groupby('question_id')
        if grp['iteration'].nunique() >= N_ITER
    )
    print(f"Resuming — completed: {len(done_ids)} / {total}")
except FileNotFoundError:
    results = []
    done_ids = set()
    print("Starting fresh.")

remaining = total - len(done_ids)
print(f"Remaining questions: {remaining}")
print(f"Remaining API calls: {remaining * N_ITER}")

# %% [markdown]
# ## 2. Run the full experiment against GPT-4o

# %%
print("\n" + "=" * 70)
print(f"RUNNING FULL v2 EXPERIMENT -- MODEL: {MODEL_UNDER_TEST}")
print("=" * 70)

for i, row in df_q.iterrows():
    q_id = row['question_id']
    question = row['question']
    evidence = row['evidence']
    gold_sql = row['SQL']
    difficulty = row['difficulty']
    category = row['category']

    if q_id in done_ids:
        print(f"[{i+1:2d}/{total}] {q_id} — skipped (done)")
        continue

    iter_correct = 0
    iter_compile_err = 0
    iter_exec_err = 0

    for iteration in range(N_ITER):
        try:
            res = query_semantic_layer_v2(question, evidence, model=MODEL_UNDER_TEST)
        except Exception as e:
            results.append({
                'question_id': q_id, 'difficulty': difficulty, 'category': category,
                'iteration': iteration, 'model': MODEL_UNDER_TEST, 'is_correct': False,
                'compile_error': None, 'execution_error': f"API call failed: {e}",
                'concept_sql': None, 'compiled_sql': None, 'gold_sql': gold_sql,
                'question': question, 'latency_ms': None, 'prompt_chars': None,
                'substitutions': None,
            })
            iter_exec_err += 1
            continue

        if res['compile_error']:
            results.append({
                'question_id': q_id, 'difficulty': difficulty, 'category': category,
                'iteration': iteration, 'model': res['model'], 'is_correct': False,
                'compile_error': res['compile_error'], 'execution_error': None,
                'concept_sql': res['concept_sql'], 'compiled_sql': None, 'gold_sql': gold_sql,
                'question': question, 'latency_ms': res['latency_ms'],
                'prompt_chars': res['prompt_chars'],
                'substitutions': ' | '.join(res['substitutions']) if res['substitutions'] else None,
            })
            iter_compile_err += 1
            continue

        ev = evaluate(res['sql'], gold_sql)
        correct = bool(ev['is_correct']) if ev['is_correct'] is not None else False
        if correct:
            iter_correct += 1
        if ev.get('error'):
            iter_exec_err += 1

        results.append({
            'question_id': q_id, 'difficulty': difficulty, 'category': category,
            'iteration': iteration, 'model': res['model'], 'is_correct': correct,
            'compile_error': None, 'execution_error': ev.get('error'),
            'concept_sql': res['concept_sql'], 'compiled_sql': res['sql'], 'gold_sql': gold_sql,
            'question': question, 'latency_ms': res['latency_ms'],
            'prompt_chars': res['prompt_chars'],
            'substitutions': ' | '.join(res['substitutions']) if res['substitutions'] else None,
        })

        time.sleep(0.3)

    pd.DataFrame(results).to_csv(RESULTS_PATH, index=False, encoding='utf-8-sig')

    df_tmp = pd.DataFrame(results)
    running_acc = df_tmp['is_correct'].mean()
    print(f"[{i+1:2d}/{total}] {q_id} ({difficulty[:3]}/{category[:8]}) "
          f"correct={iter_correct}/{N_ITER} compile_err={iter_compile_err}/{N_ITER} "
          f"exec_err={iter_exec_err}/{N_ITER} | Running accuracy: {running_acc:.1%}")

print("=" * 70)
print(f"Experiment complete. Saved: {RESULTS_PATH}")
print("=" * 70)

# %% [markdown]
# ## 3. Final summary (GPT-4o only)

# %%
df_final = pd.read_csv(RESULTS_PATH)

print(f"Total records: {len(df_final)}")
print(f"Questions: {df_final['question_id'].nunique()}")
print(f"Iterations: {df_final['iteration'].nunique()}")
print(f"Model: {df_final['model'].unique()}")

print(f"\nOverall accuracy ({MODEL_UNDER_TEST}): {df_final['is_correct'].mean():.1%}")

n_compile_err = df_final['compile_error'].notna().sum()
n_exec_err = df_final['execution_error'].notna().sum()
n_correct = df_final['is_correct'].sum()
n_wrong_result = len(df_final) - n_compile_err - n_exec_err - n_correct

print(f"\n--- Two-stage failure taxonomy ({MODEL_UNDER_TEST}) ---")
print(f"  Correct         : {n_correct} ({n_correct/len(df_final):.1%})")
print(f"  Compile errors  : {n_compile_err} ({n_compile_err/len(df_final):.1%})")
print(f"  Execution errors: {n_exec_err} ({n_exec_err/len(df_final):.1%})")
print(f"  Wrong result    : {n_wrong_result} ({n_wrong_result/len(df_final):.1%})")

print(f"\n--- Accuracy by difficulty ({MODEL_UNDER_TEST}) ---")
print(df_final.groupby('difficulty')['is_correct'].agg(['mean', 'count']))

print(f"\n--- Accuracy by category ({MODEL_UNDER_TEST}) ---")
print(df_final.groupby('category')['is_correct'].agg(['mean', 'count']))

print(f"\n--- Latency ({MODEL_UNDER_TEST}) ---")
print(df_final.groupby('question_id')['latency_ms'].mean().describe())

print("""

NEXT STEP: run 08_multi_model_comparison.ipynb to compare this
GPT-4o run against the GPT-4o-mini results
(results_phase3_v2_arch.csv), addressing Reviewer 2 Concern #2.
""")

SCRIPT VERSION: 2026-07-15-v1
DB_PATH in use: ./dev/dev_20240627/dev_databases/dev_databases/financial/financial.sqlite
Total questions: 60
Model under test: gpt-4o
Iterations per question: 5
Results will be saved to: ./results_phase3_v2_gpt4o.csv
Total API calls if run from scratch: 300
Starting fresh.
Remaining questions: 60
Remaining API calls: 300

RUNNING FULL v2 EXPERIMENT -- MODEL: gpt-4o
[ 1/60] CR01 (sim/default_) correct=1/5 compile_err=0/5 exec_err=0/5 | Running accuracy: 20.0%
[ 2/60] CR02 (sim/default_) correct=5/5 compile_err=0/5 exec_err=0/5 | Running accuracy: 60.0%
[ 3/60] CR03 (sim/default_) correct=5/5 compile_err=0/5 exec_err=0/5 | Running accuracy: 73.3%
[ 4/60] CR04 (sim/loan_por) correct=5/5 compile_err=0/5 exec_err=0/5 | Running accuracy: 80.0%
[ 5/60] CR05 (sim/loan_por) correct=5/5 compile_err=0/5 exec_err=0/5 | Running accuracy: 84.0%
[ 6/60] CR06 (sim/client_p) correct=5/5 compile_err=0/5 exec_err=0/5 | Running accuracy: 86.7%
[ 7/60] CR07 (sim/regional) cor